# R Smoke Test -- Snowflake Workspace Notebook

Minimal validation that R, **snowflakeR**, and **RSnowflake** work correctly
in a Snowflake Workspace Notebook using only the public GitHub repos:
- [snowflake-notebook-multilang](https://github.com/Snowflake-Labs/snowflake-notebook-multilang) (R bootstrap)
- [snowflakeR](https://github.com/Snowflake-Labs/snowflakeR) (ML platform: Registry, Feature Store)
- [RSnowflake](https://github.com/Snowflake-Labs/RSnowflake) (DBI connector)

**Before you start:** Optionally edit `r_smoke_test_config.yaml` to set
`context` (database/schema/warehouse) and `eai` settings. If omitted,
session defaults are used and EAIs are auto-discovered.

**Setup:**
1. Run the **Setup** cell -- it handles EAI, R bootstrap, and package install.
2. If this is a fresh setup (no EAI attached yet), follow the printed
   instructions to attach it via the Snowsight UI, then re-run the cell.
3. Once setup reports success, run the test sections.

**What this tests:**
1. R language execution via `%%R` magic
2. snowflakeR connection and query
3. snowflakeR Model Registry connectivity
4. snowflakeR Feature Store connectivity
5. RSnowflake DBI connectivity (SPCS OAuth -- no PAT needed)

**Estimated time:** ~1 min with tarballs, ~3 min from GitHub (~2 sec cached)

## 0. Setup

Single cell to set session context, validate EAI domains, install R, and
install snowflakeR + RSnowflake packages.

All configuration comes from `r_smoke_test_config.yaml`:
- **context** (optional): database, schema, warehouse overrides
- **eai** (optional): managed EAI name, supplementary EAI name
- **languages.r**: conda packages, tarballs, addons

### EAI handling

- Discovers all visible EAIs via SQL and checks domain coverage.
- If a `managed` EAI is specified in the config, missing domains are added.
- Otherwise a supplementary EAI (`MULTILANG_NOTEBOOK_EAI`) is created.
- Open (allow-all) EAIs are detected and skipped.

### First-time setup (if no EAI exists yet)

After the cell creates the EAI, attach it to your notebook service:

1. Click **Connected** (top-left of the notebook toolbar)
2. Hover over your service name and click **Edit**
3. Scroll to **External Access**
4. Toggle **ON** the EAI > **Save**
5. Service restarts automatically
6. Re-run this cell, then continue to the test sections.

Once created and attached, the same EAI can be reused across multiple
Workspace Notebooks.

In [ ]:
from sfnb_setup import setup_notebook
result = setup_notebook(config="r_smoke_test_config.yaml")

## 3. Test: R Language Execution

Verify that R code runs correctly via the `%%R` magic.

In [ ]:
%%R
cat("R version:", R.version.string, "\n")
cat("Platform: ", R.version$platform, "\n")
cat("Sum of x:  ", sum(1:5), "\n")
cat("Mean of y: ", round(mean(c(2.3, 4.1, 1.8, 5.7, 3.2)), 4), "\n")
cat("[PASS] R language execution works.\n\n")

data.frame(x = 1:5, y = c(2.3, 4.1, 1.8, 5.7, 3.2), label = letters[1:5])

## 4. Test: snowflakeR Connection & Query

Connect to Snowflake using `snowflakeR` (uses the active Workspace session
via reticulate -- no credentials needed).

In [ ]:
%%R
library(snowflakeR)

conn <- sfr_connect()
cat("[PASS] snowflakeR connection and query work.\n\n")

sfr_query(
  conn,
  "SELECT CURRENT_TIMESTAMP() AS TS, CURRENT_USER() AS WHO, CURRENT_DATABASE() AS DB"
)

## 5. Test: snowflakeR Model Registry

Connect to the Model Registry and list registered models.
An empty list is a valid result -- it confirms the Python bridge and
Snowflake ML API are working.

In [ ]:
%%R
reg <- sfr_model_registry(conn)
models <- sfr_show_models(reg)
cat("Models in registry:", nrow(models), "\n")
cat("[PASS] Model Registry connectivity works.\n\n")

if (nrow(models) > 0) models

## 6. Test: snowflakeR Feature Store

Connect to the Feature Store and list entities and feature views.
`create = TRUE` initializes Feature Store metadata in the schema if needed.
Empty results are valid -- they confirm API connectivity.

In [ ]:
%%R
fs <- sfr_feature_store(
  conn,
  database  = conn$database,
  schema    = conn$schema,
  warehouse = conn$warehouse,
  create    = TRUE
)
entities <- sfr_list_entities(fs)
fvs <- sfr_list_feature_views(fs)
cat("Entities:", nrow(entities), "\n")
cat("Feature Views:", nrow(fvs), "\n")
cat("[PASS] Feature Store connectivity works.\n\n")

if (nrow(fvs) > 0) fvs else if (nrow(entities) > 0) entities

## 7. Test: RSnowflake DBI Connectivity

RSnowflake connects via the Snowflake SQL API v2. In Workspace Notebooks
the built-in SPCS OAuth token is used automatically -- no PAT required.

The cell below sets environment variables that RSnowflake reads for
account, database, warehouse, and role context, then runs a round-trip
DBI test (connect, query, write, read, cleanup).

In [ ]:
# SPCS OAuth env vars already exported by setup_notebook()
import os
print(f"Account:   {os.environ.get('SNOWFLAKE_ACCOUNT', '?')}")
print(f"Database:  {os.environ.get('SNOWFLAKE_DATABASE', '?')}")
print(f"Warehouse: {os.environ.get('SNOWFLAKE_WAREHOUSE', '?')}")
print("Auth:      SPCS OAuth token (built-in)")

In [ ]:
%%R
if (!nzchar(Sys.getenv("TZ", ""))) Sys.setenv(TZ = "UTC")

library(RSnowflake)
library(DBI)

con <- dbConnect(Snowflake())

test_tbl <- "R_SMOKE_TEST_DBI"
dbWriteTable(con, test_tbl, mtcars[1:5, ], overwrite = TRUE)
result <- dbReadTable(con, test_tbl)
stopifnot(nrow(result) == 5)

dbRemoveTable(con, test_tbl)
dbDisconnect(con)
cat("Wrote 5 rows, read back:", nrow(result), "\n")
cat("[PASS] RSnowflake DBI round-trip works.\n\n")

result

## 8. Summary

Recap of all test results.

In [ ]:
%%R
cat("\n")
cat("============================================================\n")
cat("  R Smoke Test -- Summary\n")
cat("============================================================\n")
cat("  R version:      ", R.version.string, "\n")
cat("  snowflakeR:     ", as.character(packageVersion("snowflakeR")), "\n")
cat("  RSnowflake:     ", as.character(packageVersion("RSnowflake")), "\n")
cat("  sfnb-multilang:  (pip-installed from GitHub)\n")
cat("------------------------------------------------------------\n")
cat("  R execution:       PASS\n")
cat("  snowflakeR query:  PASS\n")
cat("  Model Registry:    PASS\n")
cat("  Feature Store:     PASS\n")
cat("  RSnowflake DBI:    PASS\n")
cat("============================================================\n")
cat("  All public packages validated successfully.\n")
cat("============================================================\n")

In [ ]:
# CI results -- writes a results table when run non-interactively
try:
    from snowflake.snowpark.context import get_active_session
    _ci_session = get_active_session()
    _ci_session.sql("""
        CREATE OR REPLACE TABLE NOTEBOOK_CI.R_SMOKE_TEST_RESULTS AS
        SELECT 'r_smoke_test' AS TEST_NAME, 'PASS' AS STATUS,
               CURRENT_TIMESTAMP() AS RUN_AT, CURRENT_USER() AS RUN_BY
    """).collect()
    print("CI results written to NOTEBOOK_CI.R_SMOKE_TEST_RESULTS")
except Exception:
    pass